# MBE Generation-Stage Hallucination Experiment — Analysis

**Model:** llama-3.3-70b-versatile (Groq free tier)  
**Classifier (FM taxonomy):** llama-3.3-70b-versatile via Groq (same model as experimental subject)  
**Dataset:** `reglab/barexam_qa` — 1,195 MBE questions (all splits)  
**Conditions:** 8 zero-shot prompting interventions × zero-shot / few-shot modes  
**Failure modes:** FM1 = Parametric Override, FM2 = Reasoning Failure (baseline errors only)

---

In [ ]:
import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
from scipy import stats
from scipy.special import expit
from statsmodels.stats.contingency_tables import mcnemar
from statsmodels.stats.proportion import proportion_confint

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', font_scale=1.1)
plt.rcParams['figure.dpi'] = 120

RESULTS_ZS  = 'results/zero_shot'
RESULTS_FS  = 'results/few_shot'
FM_PATH     = 'results/zero_shot/failure_modes_baseline.csv'

COND_NAMES = {
    0: 'Baseline',
    1: 'Grounding',
    2: 'Rule Extraction',
    3: 'Chain of Logic',
    4: 'Neg. Elimination',
    5: 'Answer Verification',
    6: 'Self-Consistency',
    7: 'Rule Ext. + CoL',
}
COND_FILES = {
    0: 'condition_0_baseline.csv',
    1: 'condition_1_grounding.csv',
    2: 'condition_2_rule_extraction.csv',
    3: 'condition_3_chain_of_logic.csv',
    4: 'condition_4_negative_elimination.csv',
    5: 'condition_5_answer_verification.csv',
    6: 'condition_6_self_consistency.csv',
    7: 'condition_7_rule_col.csv',
}
SUBJECTS = ['CONST. LAW', 'CONTRACTS', 'CRIM. LAW', 'EVIDENCE', 'REAL PROP.', 'TORTS']
COLORS = sns.color_palette('tab10', 8)

## 0. Load Data

In [ ]:
def load_mode(results_dir):
    """Load all condition CSVs from a results directory into a single DataFrame."""
    dfs = []
    for cid, fname in COND_FILES.items():
        path = os.path.join(results_dir, fname)
        if not os.path.exists(path):
            print(f'  Missing: {path}')
            continue
        df = pd.read_csv(path)
        df['condition_id'] = cid
        df['condition_name'] = COND_NAMES[cid]
        dfs.append(df)
    combined = pd.concat(dfs, ignore_index=True)
    combined['is_correct'] = combined['is_correct'].astype(bool)
    return combined

print('Loading zero-shot results...')
zs = load_mode(RESULTS_ZS)
print(f'  {len(zs):,} rows, {zs["condition_id"].nunique()} conditions')

print('Loading few-shot results...')
fs = load_mode(RESULTS_FS)
print(f'  {len(fs):,} rows, {fs["condition_id"].nunique()} conditions')

# Only keep questions present in all conditions (inner join on idx)
def filter_complete(df):
    counts = df.groupby('idx')['condition_id'].nunique()
    complete_idx = counts[counts == df['condition_id'].nunique()].index
    return df[df['idx'].isin(complete_idx)].copy()

zs_complete = filter_complete(zs)
fs_complete = filter_complete(fs)
print(f'\nComplete questions — zero-shot: {zs_complete["idx"].nunique()}, few-shot: {fs_complete["idx"].nunique()}')

# Load failure modes (classified on zero-shot baseline errors)
fm = pd.read_csv(FM_PATH) if os.path.exists(FM_PATH) else None
if fm is not None:
    print(f'Failure modes: {len(fm)} rows, FM distribution:')
    print(fm['failure_mode'].value_counts().to_dict())

## 1. Accuracy per Condition

Primary accuracy table with 95% Wilson score CIs and McNemar's test vs. baseline (Bonferroni-corrected, α=0.05/7).

In [ ]:
def wilson_ci(n_correct, n_total, alpha=0.05):
    lo, hi = proportion_confint(n_correct, n_total, alpha=alpha, method='wilson')
    return lo, hi

def accuracy_table(df, label=''):
    """Build accuracy summary with CIs and McNemar's vs baseline."""
    # Pivot: one row per question, one column per condition
    pivot = df.pivot_table(index='idx', columns='condition_id', values='is_correct', aggfunc='first')
    n = len(pivot)
    baseline = pivot[0]

    rows = []
    alpha_bonf = 0.05 / 7  # Bonferroni for 7 comparisons vs baseline
    for cid in sorted(COND_NAMES):
        if cid not in pivot.columns:
            continue
        col = pivot[cid]
        n_correct = col.sum()
        acc = n_correct / n
        lo, hi = wilson_ci(n_correct, n)
        delta = acc - (baseline.sum() / n)

        # McNemar's test vs baseline
        if cid == 0:
            p_val = np.nan
            sig = ''
        else:
            ct = pd.crosstab(baseline, col)
            # Ensure 2x2
            for v in [True, False]:
                if v not in ct.index: ct.loc[v] = 0
                if v not in ct.columns: ct[v] = 0
            ct = ct.loc[[True, False], [True, False]]
            result = mcnemar(ct.values, exact=False, correction=True)
            p_val = result.pvalue
            sig = '***' if p_val < alpha_bonf else ('*' if p_val < 0.05 else '')

        rows.append({
            'Cond': cid,
            'Name': COND_NAMES[cid],
            'N': n,
            'Accuracy': acc,
            'CI_lo': lo,
            'CI_hi': hi,
            'Δ vs Baseline': delta,
            'McNemar p': p_val,
            'Sig': sig,
        })

    tbl = pd.DataFrame(rows)
    display_cols = ['Cond', 'Name', 'N', 'Accuracy', 'CI_lo', 'CI_hi', 'Δ vs Baseline', 'McNemar p', 'Sig']
    fmt = {'Accuracy': '{:.3f}', 'CI_lo': '{:.3f}', 'CI_hi': '{:.3f}',
           'Δ vs Baseline': '{:+.3f}', 'McNemar p': '{:.4f}'}
    print(f'\n=== {label} ===' if label else '')
    return tbl

tbl_zs = accuracy_table(zs_complete, 'Zero-shot')
tbl_fs = accuracy_table(fs_complete, 'Few-shot')

print('ZERO-SHOT:')
display(tbl_zs.style.format({'Accuracy': '{:.3f}', 'CI_lo': '{:.3f}', 'CI_hi': '{:.3f}',
                              'Δ vs Baseline': '{:+.3f}', 'McNemar p': '{:.4f}'}))
print('\nFEW-SHOT:')
display(tbl_fs.style.format({'Accuracy': '{:.3f}', 'CI_lo': '{:.3f}', 'CI_hi': '{:.3f}',
                              'Δ vs Baseline': '{:+.3f}', 'McNemar p': '{:.4f}'}))

In [ ]:
# --- Plot: Accuracy per condition with CIs, zero-shot vs few-shot ---
fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=False)

for ax, tbl, title in zip(axes, [tbl_zs, tbl_fs], ['Zero-Shot', 'Few-Shot']):
    x = range(len(tbl))
    bars = ax.bar(x, tbl['Accuracy'], color=[COLORS[i] for i in tbl['Cond']],
                  alpha=0.85, edgecolor='white', linewidth=0.5)
    ax.errorbar(x, tbl['Accuracy'],
                yerr=[tbl['Accuracy'] - tbl['CI_lo'], tbl['CI_hi'] - tbl['Accuracy']],
                fmt='none', color='black', capsize=4, linewidth=1.2)
    # Baseline reference line
    baseline_acc = tbl.loc[tbl['Cond'] == 0, 'Accuracy'].values[0]
    ax.axhline(baseline_acc, color='black', linestyle='--', linewidth=1, alpha=0.6, label=f'Baseline ({baseline_acc:.3f})')
    ax.set_xticks(x)
    ax.set_xticklabels(tbl['Name'], rotation=35, ha='right', fontsize=9)
    ax.set_ylabel('Accuracy')
    ax.set_title(title)
    ax.yaxis.set_major_formatter(mtick.PercentFormatter(xmax=1))
    ax.legend(fontsize=8)
    # Significance markers
    for xi, (_, row) in zip(x, tbl.iterrows()):
        if row['Sig']:
            ax.text(xi, row['CI_hi'] + 0.005, row['Sig'], ha='center', fontsize=10)

plt.suptitle('Accuracy per Condition (95% Wilson CI, * p<0.05, *** Bonferroni-corrected)', fontsize=11)
plt.tight_layout()
plt.savefig('results/analysis/accuracy_per_condition.png', bbox_inches='tight')
plt.show()

## 2. Regression Matrix — Gains vs. Losses vs. Baseline

For each condition: how many questions did it gain (wrong→right) vs. lose (right→wrong) vs. baseline?

In [ ]:
def regression_matrix(df, label=''):
    pivot = df.pivot_table(index='idx', columns='condition_id', values='is_correct', aggfunc='first')
    baseline = pivot[0]
    rows = []
    for cid in sorted(COND_NAMES):
        if cid == 0 or cid not in pivot.columns:
            continue
        col = pivot[cid]
        both_right   = ((baseline == True)  & (col == True)).sum()
        gained       = ((baseline == False) & (col == True)).sum()   # wrong→right
        lost         = ((baseline == True)  & (col == False)).sum()  # right→wrong
        both_wrong   = ((baseline == False) & (col == False)).sum()
        net = gained - lost
        rows.append({'Cond': cid, 'Name': COND_NAMES[cid],
                     'Both Correct': both_right, 'Gained (↑)': gained,
                     'Lost (↓)': lost, 'Both Wrong': both_wrong, 'Net': net})
    return pd.DataFrame(rows)

rm_zs = regression_matrix(zs_complete)
rm_fs = regression_matrix(fs_complete)

print('ZERO-SHOT regression matrix:')
display(rm_zs)
print('\nFEW-SHOT regression matrix:')
display(rm_fs)

In [ ]:
# Plot: Gains vs Losses as diverging bars
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, rm, title in zip(axes, [rm_zs, rm_fs], ['Zero-Shot', 'Few-Shot']):
    y = range(len(rm))
    ax.barh(y, rm['Gained (↑)'], color='steelblue', alpha=0.85, label='Gained (wrong→right)')
    ax.barh(y, -rm['Lost (↓)'],  color='tomato',    alpha=0.85, label='Lost (right→wrong)')
    ax.axvline(0, color='black', linewidth=0.8)
    for yi, (_, row) in zip(y, rm.iterrows()):
        ax.text(row['Gained (↑)'] + 1, yi, f"+{row['Gained (↑)']}", va='center', fontsize=8, color='steelblue')
        ax.text(-row['Lost (↓)'] - 1, yi, f"-{row['Lost (↓)']}", va='center', ha='right', fontsize=8, color='tomato')
        ax.text(0, yi + 0.35, f"net: {row['Net']:+d}", va='bottom', ha='center', fontsize=7.5, color='black')
    ax.set_yticks(y)
    ax.set_yticklabels(rm['Name'])
    ax.set_xlabel('Questions')
    ax.set_title(title)
    ax.legend(fontsize=8)

plt.suptitle('Gains vs. Losses vs. Baseline per Condition', fontsize=11)
plt.tight_layout()
plt.savefig('results/analysis/regression_matrix.png', bbox_inches='tight')
plt.show()

## 3. Cost-Accuracy Pareto Frontier

Accuracy vs. average tokens per question. Identifies which conditions are worth their cost.

In [ ]:
def cost_accuracy(df, label=''):
    df['total_tokens'] = df['tokens_input'].fillna(0) + df['tokens_output'].fillna(0)
    summary = df.groupby(['condition_id', 'condition_name']).agg(
        accuracy=('is_correct', 'mean'),
        avg_tokens=('total_tokens', 'mean'),
    ).reset_index()
    return summary

ca_zs = cost_accuracy(zs_complete)
ca_fs = cost_accuracy(fs_complete)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, ca, title in zip(axes, [ca_zs, ca_fs], ['Zero-Shot', 'Few-Shot']):
    for _, row in ca.iterrows():
        cid = int(row['condition_id'])
        ax.scatter(row['avg_tokens'], row['accuracy'], s=120, color=COLORS[cid],
                   zorder=5, edgecolors='white', linewidths=0.8)
        ax.annotate(row['condition_name'], (row['avg_tokens'], row['accuracy']),
                    textcoords='offset points', xytext=(6, 4), fontsize=8)
    ax.set_xlabel('Avg Tokens per Question (input + output)')
    ax.set_ylabel('Accuracy')
    ax.set_title(title)
    ax.yaxis.set_major_formatter(mtick.PercentFormatter(xmax=1))

plt.suptitle('Cost-Accuracy Pareto Frontier', fontsize=11)
plt.tight_layout()
plt.savefig('results/analysis/cost_accuracy_pareto.png', bbox_inches='tight')
plt.show()

## 4. Zero-Shot vs. Few-Shot × Condition

Does few-shot amplify or dampen the effect of each condition?

In [ ]:
# Only use questions present in both modes
common_idx = set(zs_complete['idx'].unique()) & set(fs_complete['idx'].unique())
print(f'Questions in both modes: {len(common_idx)}')

zs_common = zs_complete[zs_complete['idx'].isin(common_idx)]
fs_common = fs_complete[fs_complete['idx'].isin(common_idx)]

acc_zs = zs_common.groupby('condition_id')['is_correct'].mean().rename('zero_shot')
acc_fs = fs_common.groupby('condition_id')['is_correct'].mean().rename('few_shot')
mode_df = pd.concat([acc_zs, acc_fs], axis=1).reset_index()
mode_df['condition_name'] = mode_df['condition_id'].map(COND_NAMES)
mode_df['few_shot_delta'] = mode_df['few_shot'] - mode_df['zero_shot']

display(mode_df.style.format({'zero_shot': '{:.3f}', 'few_shot': '{:.3f}', 'few_shot_delta': '{:+.3f}'}))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Grouped bar chart
ax = axes[0]
x = np.arange(len(mode_df))
w = 0.35
ax.bar(x - w/2, mode_df['zero_shot'], w, label='Zero-Shot', alpha=0.85, color='steelblue')
ax.bar(x + w/2, mode_df['few_shot'],  w, label='Few-Shot',  alpha=0.85, color='darkorange')
ax.set_xticks(x)
ax.set_xticklabels(mode_df['condition_name'], rotation=35, ha='right', fontsize=8)
ax.set_ylabel('Accuracy')
ax.set_title('Accuracy: Zero-Shot vs. Few-Shot')
ax.yaxis.set_major_formatter(mtick.PercentFormatter(xmax=1))
ax.legend()

# Delta plot
ax = axes[1]
colors = ['steelblue' if d >= 0 else 'tomato' for d in mode_df['few_shot_delta']]
ax.bar(x, mode_df['few_shot_delta'], color=colors, alpha=0.85)
ax.axhline(0, color='black', linewidth=0.8)
ax.set_xticks(x)
ax.set_xticklabels(mode_df['condition_name'], rotation=35, ha='right', fontsize=8)
ax.set_ylabel('Few-Shot Δ Accuracy')
ax.set_title('Few-Shot Gain per Condition (vs. Zero-Shot)')
ax.yaxis.set_major_formatter(mtick.PercentFormatter(xmax=1))

plt.suptitle('Zero-Shot vs. Few-Shot × Condition', fontsize=11)
plt.tight_layout()
plt.savefig('results/analysis/zs_vs_fs_by_condition.png', bbox_inches='tight')
plt.show()

## 5. Subject × Condition Heatmap

Accuracy per MBE subject × condition. Delta vs. baseline shown alongside raw accuracy.

In [ ]:
def subject_heatmap(df, title=''):
    df_subj = df[df['subject'].isin(SUBJECTS)].copy()
    pivot = df_subj.pivot_table(
        index='subject', columns='condition_id', values='is_correct', aggfunc='mean'
    )
    pivot.columns = [COND_NAMES[c] for c in pivot.columns]
    # Delta vs baseline
    delta = pivot.sub(pivot['Baseline'], axis=0)

    fig, axes = plt.subplots(1, 2, figsize=(16, 5))

    sns.heatmap(pivot, annot=True, fmt='.2f', cmap='RdYlGn',
                vmin=0.45, vmax=0.95, linewidths=0.5, ax=axes[0],
                cbar_kws={'label': 'Accuracy'})
    axes[0].set_title(f'{title} — Raw Accuracy')
    axes[0].set_xlabel('')
    axes[0].set_xticklabels(axes[0].get_xticklabels(), rotation=35, ha='right')

    sns.heatmap(delta, annot=True, fmt='+.2f', cmap='RdYlGn',
                center=0, vmin=-0.15, vmax=0.15, linewidths=0.5, ax=axes[1],
                cbar_kws={'label': 'Δ vs Baseline'})
    axes[1].set_title(f'{title} — Δ vs Baseline')
    axes[1].set_xlabel('')
    axes[1].set_xticklabels(axes[1].get_xticklabels(), rotation=35, ha='right')

    plt.tight_layout()
    fname = title.lower().replace(' ', '_').replace('-', '_')
    plt.savefig(f'results/analysis/subject_heatmap_{fname}.png', bbox_inches='tight')
    plt.show()
    return pivot, delta

pivot_zs, delta_zs = subject_heatmap(zs_complete, 'Zero-Shot')
pivot_fs, delta_fs = subject_heatmap(fs_complete, 'Few-Shot')

## 6. Failure Mode × Condition

For each baseline error classified as FM1 (Parametric Override) or FM2 (Reasoning Failure),
what fraction does each condition fix? Tests whether interventions target the right failure mode.

In [ ]:
if fm is None:
    print('No failure modes CSV found — skipping FM analysis.')
else:
    # Merge FM labels into zero-shot results (FM only applies to baseline errors)
    zs_with_fm = zs_complete.merge(fm[['idx', 'failure_mode']], on='idx', how='left')

    # For each condition, on the subset of questions that baseline got WRONG
    # (classified FM1 or FM2), what fraction does each condition now get right?
    baseline_errors = zs_with_fm[zs_with_fm['condition_id'] == 0][['idx', 'failure_mode']]
    baseline_errors = baseline_errors[baseline_errors['failure_mode'].isin(['FM1', 'FM2'])]

    rows = []
    for cid in sorted(COND_NAMES):
        cond_df = zs_with_fm[zs_with_fm['condition_id'] == cid]
        for fm_type in ['FM1', 'FM2']:
            fm_idx = baseline_errors[baseline_errors['failure_mode'] == fm_type]['idx']
            subset = cond_df[cond_df['idx'].isin(fm_idx)]
            if len(subset) == 0:
                continue
            fix_rate = subset['is_correct'].mean()
            rows.append({'condition_id': cid, 'condition_name': COND_NAMES[cid],
                         'failure_mode': fm_type, 'fix_rate': fix_rate, 'n': len(subset)})

    fm_df = pd.DataFrame(rows)
    display(fm_df.pivot_table(index='condition_name', columns='failure_mode',
                               values='fix_rate').style.format('{:.3f}').background_gradient(cmap='RdYlGn'))

    # Plot
    fig, ax = plt.subplots(figsize=(10, 5))
    fm_pivot = fm_df.pivot_table(index='condition_id', columns='failure_mode', values='fix_rate')
    x = np.arange(len(fm_pivot))
    w = 0.35
    ax.bar(x - w/2, fm_pivot.get('FM1', 0), w, label='FM1 (Parametric Override)', color='steelblue', alpha=0.85)
    ax.bar(x + w/2, fm_pivot.get('FM2', 0), w, label='FM2 (Reasoning Failure)',   color='darkorange', alpha=0.85)
    ax.set_xticks(x)
    ax.set_xticklabels([COND_NAMES[i] for i in fm_pivot.index], rotation=35, ha='right')
    ax.set_ylabel('Fix Rate (fraction of baseline errors now correct)')
    ax.set_title('Failure Mode Fix Rate per Condition (Zero-Shot)')
    ax.yaxis.set_major_formatter(mtick.PercentFormatter(xmax=1))
    ax.legend()
    plt.tight_layout()
    plt.savefig('results/analysis/fm_by_condition.png', bbox_inches='tight')
    plt.show()

## 7. Condition 6 — Self-Consistency Agreement Rate

Does unanimous 3/3 agreement predict correctness better than 2/1 split?

In [ ]:
for df, label in [(zs_complete, 'Zero-Shot'), (fs_complete, 'Few-Shot')]:
    cond6 = df[df['condition_id'] == 6].copy()
    if cond6.empty:
        print(f'{label}: no Condition 6 data')
        continue

    # confidence = votes_for_winner / 3 → infer vote count
    cond6['votes'] = (cond6['confidence'] * 3).round().astype(int)
    cond6['unanimous'] = cond6['votes'] == 3

    summary = cond6.groupby('unanimous').agg(
        n=('is_correct', 'count'),
        accuracy=('is_correct', 'mean')
    ).reset_index()
    summary['unanimous'] = summary['unanimous'].map({True: '3/3 unanimous', False: '2/1 split'})

    print(f'\n{label} — Condition 6 agreement:')
    display(summary)

    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    axes[0].pie(summary['n'], labels=summary['unanimous'], autopct='%1.1f%%',
                colors=['steelblue', 'darkorange'])
    axes[0].set_title('Vote Agreement Distribution')
    axes[1].bar(summary['unanimous'], summary['accuracy'],
                color=['steelblue', 'darkorange'], alpha=0.85)
    axes[1].set_ylabel('Accuracy')
    axes[1].set_title('Accuracy by Agreement Level')
    axes[1].yaxis.set_major_formatter(mtick.PercentFormatter(xmax=1))
    plt.suptitle(f'Condition 6 Self-Consistency — {label}', fontsize=11)
    plt.tight_layout()
    plt.savefig(f'results/analysis/cond6_agreement_{label.lower().replace("-", "_").replace(" ", "_")}.png', bbox_inches='tight')
    plt.show()

## 8. Confidence Calibration per Failure Mode

Are FM1 (Parametric Override) errors more overconfident than FM2 (Reasoning Failure)?

In [ ]:
if fm is None:
    print('No failure modes CSV — skipping.')
else:
    # Use zero-shot baseline (condition 0)
    baseline_df = zs_complete[zs_complete['condition_id'] == 0].copy()
    baseline_df = baseline_df.merge(fm[['idx', 'failure_mode']], on='idx', how='left')

    # Confidence distribution: correct, FM1 errors, FM2 errors
    correct   = baseline_df[baseline_df['is_correct'] == True]['confidence'].dropna()
    fm1_errors = baseline_df[(baseline_df['is_correct'] == False) &
                              (baseline_df['failure_mode'] == 'FM1')]['confidence'].dropna()
    fm2_errors = baseline_df[(baseline_df['is_correct'] == False) &
                              (baseline_df['failure_mode'] == 'FM2')]['confidence'].dropna()

    print(f'Correct   — mean confidence: {correct.mean():.3f} (n={len(correct)})')
    print(f'FM1 errors — mean confidence: {fm1_errors.mean():.3f} (n={len(fm1_errors)})')
    print(f'FM2 errors — mean confidence: {fm2_errors.mean():.3f} (n={len(fm2_errors)})')

    # Mann-Whitney U: FM1 vs FM2 confidence
    if len(fm1_errors) > 0 and len(fm2_errors) > 0:
        u, p = stats.mannwhitneyu(fm1_errors, fm2_errors, alternative='two-sided')
        print(f'\nMann-Whitney FM1 vs FM2 confidence: U={u:.0f}, p={p:.4f}')

    fig, axes = plt.subplots(1, 2, figsize=(13, 5))

    # KDE
    ax = axes[0]
    for data, lbl, color in [(correct, 'Correct', 'steelblue'),
                              (fm1_errors, 'FM1 (Parametric Override)', 'tomato'),
                              (fm2_errors, 'FM2 (Reasoning Failure)', 'darkorange')]:
        if len(data) > 1:
            data.plot.kde(ax=ax, label=f'{lbl} (μ={data.mean():.2f})', color=color)
    ax.set_xlabel('Confidence')
    ax.set_title('Confidence Distribution by Outcome / FM Type')
    ax.legend(fontsize=8)
    ax.set_xlim(0, 1)

    # Box plot
    ax = axes[1]
    data_box = [correct, fm1_errors, fm2_errors]
    labels_box = ['Correct', 'FM1 Errors', 'FM2 Errors']
    bp = ax.boxplot(data_box, labels=labels_box, patch_artist=True,
                    medianprops={'color': 'black', 'linewidth': 2})
    for patch, color in zip(bp['boxes'], ['steelblue', 'tomato', 'darkorange']):
        patch.set_facecolor(color)
        patch.set_alpha(0.7)
    ax.set_ylabel('Confidence')
    ax.set_title('Confidence by Outcome / FM Type')

    plt.suptitle('Calibration per Failure Mode (Baseline, Zero-Shot)', fontsize=11)
    plt.tight_layout()
    plt.savefig('results/analysis/calibration_per_fm.png', bbox_inches='tight')
    plt.show()

## 9. Compound Condition Decomposition

Does Condition 7 (Rule Extraction + CoL) outperform Condition 2 (Rule Extraction) and Condition 3 (CoL) individually? Checks whether combining interventions helps beyond either alone.

In [ ]:
for df, label in [(zs_complete, 'Zero-Shot'), (fs_complete, 'Few-Shot')]:
    subset = df[df['condition_id'].isin([0, 2, 3, 7])].copy()
    if subset['condition_id'].nunique() < 4:
        print(f'{label}: incomplete data for decomposition')
        continue

    acc = subset.groupby('condition_id')['is_correct'].mean()
    print(f'\n{label}:')
    for cid in [0, 2, 3, 7]:
        print(f'  Cond {cid} {COND_NAMES[cid]}: {acc.get(cid, np.nan):.3f}')

    # Per-question 2x2: does 7 fix what 2 and 3 individually missed?
    pivot = df[df['condition_id'].isin([2, 3, 7])].pivot_table(
        index='idx', columns='condition_id', values='is_correct', aggfunc='first'
    )
    if 2 in pivot.columns and 3 in pivot.columns and 7 in pivot.columns:
        # Questions where 7 is right but both 2 and 3 are wrong
        synergy = ((pivot[7] == True) & (pivot[2] == False) & (pivot[3] == False)).sum()
        # Questions where 7 is wrong but at least one of 2/3 is right
        interference = ((pivot[7] == False) & ((pivot[2] == True) | (pivot[3] == True))).sum()
        print(f'  Synergy (7 right, 2+3 both wrong): {synergy}')
        print(f'  Interference (7 wrong, 2 or 3 right): {interference}')

# Plot
fig, ax = plt.subplots(figsize=(8, 5))
cids = [0, 2, 3, 7]
x = np.arange(len(cids))
w = 0.35

for i, (df, label, color) in enumerate([(zs_complete, 'Zero-Shot', 'steelblue'),
                                          (fs_complete, 'Few-Shot', 'darkorange')]):
    acc = df[df['condition_id'].isin(cids)].groupby('condition_id')['is_correct'].mean()
    vals = [acc.get(c, np.nan) for c in cids]
    ax.bar(x + (i - 0.5) * w, vals, w, label=label, color=color, alpha=0.85)

ax.set_xticks(x)
ax.set_xticklabels([f'Cond {c}\n{COND_NAMES[c]}' for c in cids])
ax.set_ylabel('Accuracy')
ax.set_title('Compound Condition Decomposition: Cond 2 vs 3 vs 7')
ax.yaxis.set_major_formatter(mtick.PercentFormatter(xmax=1))
ax.legend()
plt.tight_layout()
plt.savefig('results/analysis/compound_decomposition.png', bbox_inches='tight')
plt.show()

## 10. Hard-Core Failures

Questions wrong across ALL 8 conditions. What do they have in common (subject distribution, FM type)?

In [ ]:
for df, label in [(zs_complete, 'Zero-Shot'), (fs_complete, 'Few-Shot')]:
    pivot = df.pivot_table(index='idx', columns='condition_id', values='is_correct', aggfunc='first')
    # Questions wrong in every condition
    hardcore = pivot[pivot.apply(lambda row: not row.any(), axis=1)].index
    print(f'\n{label} — Hard-core failures (wrong in all conditions): {len(hardcore)}')
    if len(hardcore) == 0:
        continue

    meta = df[df['idx'].isin(hardcore) & (df['condition_id'] == 0)][['idx', 'subject', 'source']].drop_duplicates()

    print('  Subject distribution:')
    print(meta['subject'].value_counts().to_dict())

    if fm is not None:
        fm_hardcore = fm[fm['idx'].isin(hardcore)]
        print('  Failure mode distribution:')
        print(fm_hardcore['failure_mode'].value_counts().to_dict())

# Visualise subject distribution for zero-shot hard-core
pivot_zs = zs_complete.pivot_table(index='idx', columns='condition_id', values='is_correct', aggfunc='first')
hardcore_zs = pivot_zs[pivot_zs.apply(lambda row: not row.any(), axis=1)].index
meta_zs = zs_complete[zs_complete['idx'].isin(hardcore_zs) & (zs_complete['condition_id'] == 0)]

# Compare subject proportions: hard-core vs overall
overall_subj = zs_complete[zs_complete['condition_id'] == 0]['subject'].value_counts(normalize=True)
hc_subj = meta_zs['subject'].value_counts(normalize=True)

compare = pd.DataFrame({'overall': overall_subj, 'hard_core': hc_subj}).fillna(0)

fig, ax = plt.subplots(figsize=(9, 4))
x = np.arange(len(compare))
w = 0.35
ax.bar(x - w/2, compare['overall'],   w, label='All Questions', color='steelblue', alpha=0.85)
ax.bar(x + w/2, compare['hard_core'], w, label='Hard-Core Failures', color='tomato', alpha=0.85)
ax.set_xticks(x)
ax.set_xticklabels(compare.index, rotation=20, ha='right')
ax.set_ylabel('Proportion')
ax.set_title('Subject Distribution: All Questions vs. Hard-Core Failures (Zero-Shot)')
ax.yaxis.set_major_formatter(mtick.PercentFormatter(xmax=1))
ax.legend()
plt.tight_layout()
plt.savefig('results/analysis/hardcore_failures_subject.png', bbox_inches='tight')
plt.show()

## 11. Summary Table

All key metrics in one place.

In [ ]:
for path, label in [('results/zero_shot/summary.csv', 'Zero-Shot'),
                     ('results/few_shot/summary.csv', 'Few-Shot')]:
    if not os.path.exists(path):
        continue
    s = pd.read_csv(path)
    print(f'\n=== {label} Summary ===')
    display(s[['condition_id', 'condition_name', 'accuracy', 'ece',
                'mean_confidence', 'overconfidence_gap', 'estimated_cost_usd']]
            .style.format({'accuracy': '{:.3f}', 'ece': '{:.4f}',
                           'mean_confidence': '{:.3f}', 'overconfidence_gap': '{:+.3f}',
                           'estimated_cost_usd': '${:.4f}'}))